# 🧪 Lasmoid 10M — Full Validation & Training

**Goal: Get a clear GO / NO-GO confirmation before scaling to bigger models.**

This notebook runs a complete diagnostic of every Lasmoid subsystem, trains a 10M model,
and produces a final verdict. Every step logs detailed status so you can see exactly
what works and what fails.

### Test phases:
| # | Phase | What it checks |
|---|-------|----------------|
| 1 | Environment | GPU, deps, repo |
| 2 | Model Build | Architecture instantiates |
| 3 | Forward Pass | All subsystems produce valid shapes |
| 4 | Subsystem Probe | Attention, SSM, mHC, MoE, VQ, CIF, MTP individually |
| 5 | Gradient Flow | Backward pass, finite grads, no dead params |
| 6 | Determinism | Same seed → same output |
| 7 | Overfit Test | Model can memorize a tiny batch (proves learning works) |
| 8 | Full Training | 5000 steps on TinyStories |
| 9 | Generation | Produces coherent English |
| 10 | **VERDICT** | GO / NO-GO for scaling |

**Run all cells top-to-bottom. No edits required.**

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1: Install dependencies
# ════════════════════════════════════════════════════════════════
# IMPORTANT: Kaggle/Colab ship with GPU-compatible torch & transformers.
# Reinstalling torch breaks the CUDA kernel match (cudaErrorNoKernelImage).
# Only install what's missing.
!pip install -q safetensors tiktoken datasets huggingface_hub

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2: Clone repo
# ════════════════════════════════════════════════════════════════
import os
WORK_DIR = '/kaggle/working/Lasmoid'

if not os.path.exists(os.path.join(WORK_DIR, 'inference', 'model.py')):
    !git clone https://github.com/Theory903/Lasmoid.git {WORK_DIR}
else:
    print('✅ Already cloned')
os.chdir(WORK_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3: Diagnostic logger + global results tracker
# ════════════════════════════════════════════════════════════════
import sys, json, time, traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, os.path.join(WORK_DIR, 'inference'))
sys.path.insert(0, os.path.join(WORK_DIR, 'train'))
sys.path.insert(0, WORK_DIR)

# ── Global results tracker for final verdict ──
RESULTS = {}

def check(name, condition, detail=''):
    """Record a pass/fail check with logging."""
    status = 'PASS' if condition else 'FAIL'
    icon = '✅' if condition else '❌'
    RESULTS[name] = condition
    print(f'  {icon} [{status}] {name}' + (f'  →  {detail}' if detail else ''))
    return condition

def section(title):
    print('\n' + '═' * 64)
    print(f'  {title}')
    print('═' * 64)

def safe_run(fn, name):
    """Run a function, catch errors, record result."""
    try:
        return fn()
    except Exception as e:
        RESULTS[name] = False
        print(f'  ❌ [ERROR] {name}')
        print(f'     {type(e).__name__}: {e}')
        traceback.print_exc()
        return None

print('✅ Logger ready')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 1: Environment Check
# ════════════════════════════════════════════════════════════════
section('PHASE 1: ENVIRONMENT')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
check('PyTorch installed', True, torch.__version__)
check('GPU available', DEVICE == 'cuda',
      torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only (slower)')
if DEVICE == 'cuda':
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    check('GPU memory >= 4GB', mem >= 4, f'{mem:.1f} GB')

check('inference/model.py exists', os.path.exists(os.path.join(WORK_DIR, 'inference', 'model.py')))
check('config_10m.json exists', os.path.exists(os.path.join(WORK_DIR, 'config_10m.json')))
check('tokenizer.json exists', os.path.exists(os.path.join(WORK_DIR, 'tokenizer.json')))

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 2: Model Build
# ════════════════════════════════════════════════════════════════
section('PHASE 2: MODEL BUILD')

from dataclasses import fields as dc_fields
import transformers
from model import Lasmoid, ModelArgs, compute_loss
from optimizer import Muon, build_optimizers
from scheduler import WSDScheduler
check('Imports succeed', True)

with open(os.path.join(WORK_DIR, 'config_10m.json')) as f:
    cfg = json.load(f)
valid = {f.name for f in dc_fields(ModelArgs)}
args = ModelArgs(**{k: v for k, v in cfg.items() if k in valid})
check('Config loaded', True, f'dim={args.dim}, layers={args.n_layers}')

# Tokenizer (try with/without the regex flag for version compatibility)
try:
    enc = transformers.PreTrainedTokenizerFast.from_pretrained(WORK_DIR)
except Exception:
    enc = transformers.PreTrainedTokenizerFast.from_pretrained(WORK_DIR, fix_mistral_regex=True)
args.vocab_size = max(args.vocab_size, len(enc))
EOS_ID = enc.eos_token_id or 1
SEQ_LEN = args.max_seq_len
check('Tokenizer loaded', True, f'vocab={len(enc)}')

torch.manual_seed(42)
model = Lasmoid(args).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
check('Model built', True, f'{n_params:.1f}M params')
check('Params in target range (5-20M)', 5 <= n_params <= 20, f'{n_params:.1f}M')
print(f'\n  Architecture: dim={args.dim} layers={args.n_layers} heads={args.n_heads} '
      f'experts={args.n_routed_experts} streams={args.num_residual_streams}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 3: Forward Pass
# ════════════════════════════════════════════════════════════════
section('PHASE 3: FORWARD PASS')

model.eval()
B, S = 2, SEQ_LEN
x = torch.randint(0, args.vocab_size, (B, S), device=DEVICE)

def run_forward():
    with torch.no_grad():
        return model(x, x)

out = safe_run(run_forward, 'Forward pass runs')

if out is not None:
    logits, mtp_logits, concept_db, memory_state, rmaps, indices, adjs, eprobs = out
    check('Forward pass runs', True)
    check('Logits shape correct', logits.shape == (B, S, args.vocab_size),
          f'{tuple(logits.shape)}')
    check('Logits are finite', torch.isfinite(logits).all().item())
    check('MTP head present', mtp_logits is not None,
          f'{tuple(mtp_logits.shape)}' if mtp_logits is not None else 'None')
    check('Concept memory present', concept_db is not None,
          f'{tuple(concept_db.shape)}' if concept_db is not None else 'None')
    check('MoE routing maps present', len(rmaps) > 0, f'{len(rmaps)} layers')
    check('VQ adjacencies present', len(adjs) > 0)
    check('CIF event probs present', eprobs is not None and len(eprobs) > 0)

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 4: Per-Subsystem Probe
# ════════════════════════════════════════════════════════════════
section('PHASE 4: SUBSYSTEM PROBE')

# Probe each subsystem in isolation using a random hidden state
torch.manual_seed(1234)
h = torch.randn(2, SEQ_LEN, args.dim, device=DEVICE)

# ── RMSNorm ──
def probe_norm():
    from _common import RMSNorm
    norm = RMSNorm(args.dim, args.norm_eps).to(DEVICE)
    o = norm(h)
    # RMSNorm output should have ~unit RMS per position
    rms = o.float().square().mean(-1).sqrt().mean().item()
    return check('RMSNorm produces normalized output', 0.5 < rms < 2.0, f'mean RMS={rms:.3f}')
safe_run(probe_norm, 'RMSNorm')

# ── RoPE ──
def probe_rope():
    from _common import apply_rotary_emb
    from attention import precompute_freqs_cis
    fc = precompute_freqs_cis(args.rope_head_dim, SEQ_LEN, 0, args.rope_theta, 1.0, 32, 1).to(DEVICE)
    q = torch.randn(2, SEQ_LEN, args.n_heads, args.rope_head_dim, device=DEVICE)
    rotated = apply_rotary_emb(q, fc)
    return check('RoPE preserves shape & is finite',
                 rotated.shape == q.shape and torch.isfinite(rotated).all().item())
safe_run(probe_rope, 'RoPE')

# ── SSM ──
def probe_ssm():
    layer = model.layers[0]
    ssm = layer.ssm_branch if hasattr(layer, 'ssm_branch') else None
    return check('SSM branch exists in block', ssm is not None)
safe_run(probe_ssm, 'SSM')

# ── MoE expert reachability ──
def probe_moe():
    from moe import Gate
    gate = Gate(0, args).to(DEVICE)
    gate.eval()
    flat = torch.randn(256, args.dim, device=DEVICE)
    with torch.no_grad():
        w, idx, zloss, probs = gate(flat)
    activated = set(idx.unique().tolist())
    coverage = len(activated) / args.n_routed_experts
    wsum = w.sum(-1).mean().item()
    check('MoE gate renormalizes (~1.0)', abs(wsum - 1.0) < 0.1 or abs(wsum - args.n_routed_experts) < 1.0,
          f'weight sum={wsum:.3f}')
    return check('MoE all experts reachable', coverage >= 0.99,
                 f'{len(activated)}/{args.n_routed_experts} experts hit')
safe_run(probe_moe, 'MoE')

# ── mHC doubly-stochastic ──
def probe_mhc():
    from mhc import ManifoldConstrainedHyperConnection
    n_hc = args.num_residual_streams
    if n_hc == 1:
        return check('mHC single-stream (identity)', True, 'n_hc=1, plain residual')
    mhc = ManifoldConstrainedHyperConnection(args.dim, n_hc, args.hc_sinkhorn_iters).to(DEVICE)
    mhc.eval()
    xx = torch.randn(2, 8, n_hc, args.dim, device=DEVICE)
    with torch.no_grad():
        A, Bmix, C = mhc(xx)
    row = (Bmix.sum(-1) - 1.0).abs().max().item()
    col = (Bmix.sum(-2) - 1.0).abs().max().item()
    return check('mHC Sinkhorn doubly-stochastic', max(row, col) < 1e-2,
                 f'row dev={row:.1e}, col dev={col:.1e}')
safe_run(probe_mhc, 'mHC')

# ── VQ ──
def probe_vq():
    from vq import VectorQuantizer
    vq = VectorQuantizer(args.codebook_size, args.dim).to(DEVICE)
    z = torch.randn(2, 8, args.dim, device=DEVICE)
    q, loss, idx = vq(z)
    return check('VQ quantizes & computes loss', q.shape == z.shape and torch.isfinite(loss).item(),
                 f'loss={loss.item():.4f}')
safe_run(probe_vq, 'VQ')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 5: Gradient Flow
# ════════════════════════════════════════════════════════════════
section('PHASE 5: GRADIENT FLOW')

model.train()
model.zero_grad()
x = torch.randint(0, args.vocab_size, (2, SEQ_LEN), device=DEVICE)
y = torch.randint(0, args.vocab_size, (2, SEQ_LEN), device=DEVICE)

def grad_test():
    logits, mtp_logits, _, _, rmaps, _, adjs, eprobs = model(x, x)
    loss = compute_loss(logits, y, rmaps, [model.last_vq_loss], adjs, eprobs,
                        moe_aux_loss=model.last_moe_loss, ignore_index=-100)
    loss.backward()
    return loss

loss = safe_run(grad_test, 'Backward pass runs')

if loss is not None:
    check('Loss is finite', torch.isfinite(loss).item(), f'loss={loss.item():.4f}')
    
    # Count params with/without gradients
    total, with_grad, nonzero_grad, nonfinite = 0, 0, 0, 0
    dead_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        total += 1
        if p.grad is not None:
            with_grad += 1
            if not torch.isfinite(p.grad).all():
                nonfinite += 1
            if p.grad.abs().sum() > 0:
                nonzero_grad += 1
            else:
                dead_params.append(name)
    
    pct_grad = with_grad / total * 100
    pct_nonzero = nonzero_grad / total * 100
    check('>80% params receive gradients', pct_grad > 80, f'{pct_grad:.0f}% ({with_grad}/{total})')
    check('>70% params have nonzero grad', pct_nonzero > 70, f'{pct_nonzero:.0f}%')
    check('No non-finite gradients', nonfinite == 0, f'{nonfinite} non-finite')
    if dead_params and len(dead_params) <= 10:
        print(f'\n  ℹ️  Params with zero grad (may be conditional paths):')
        for d in dead_params[:10]:
            print(f'      - {d}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 6: Determinism
# ════════════════════════════════════════════════════════════════
section('PHASE 6: DETERMINISM')

model.eval()
xd = torch.randint(0, args.vocab_size, (1, SEQ_LEN), device=DEVICE)

def determinism_test():
    torch.manual_seed(7)
    with torch.no_grad():
        o1 = model(xd, xd)[0]
    torch.manual_seed(7)
    with torch.no_grad():
        o2 = model(xd, xd)[0]
    max_diff = (o1 - o2).abs().max().item()
    return check('Deterministic under fixed seed', max_diff < 1e-4, f'max|Δ|={max_diff:.2e}')

safe_run(determinism_test, 'Determinism')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 7: Overfit Test (CRITICAL — proves the model can learn)
# ════════════════════════════════════════════════════════════════
section('PHASE 7: OVERFIT TEST')
print('  Training on a single fixed batch — loss MUST drop sharply if learning works.\n')

model.train()
overfit_opts = build_optimizers(model, muon_lr=3e-3, adamw_lr=1e-3, weight_decay=0.0)

# Fixed tiny batch
torch.manual_seed(0)
fx = torch.randint(0, args.vocab_size, (2, SEQ_LEN), device=DEVICE)
fy = torch.randint(0, args.vocab_size, (2, SEQ_LEN), device=DEVICE)

overfit_losses = []
def overfit():
    for i in range(60):
        for opt in overfit_opts:
            opt.zero_grad(set_to_none=True)
        logits, _, _, _, rmaps, _, adjs, eprobs = model(fx, fx)
        l = compute_loss(logits, fy, rmaps, [model.last_vq_loss], adjs, eprobs,
                         moe_aux_loss=model.last_moe_loss, ignore_index=-100)
        l.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        for opt in overfit_opts:
            opt.step()
        overfit_losses.append(l.item())
        if i % 15 == 0:
            print(f'    iter {i:2d} | loss {l.item():.3f}')
    return overfit_losses

safe_run(overfit, 'Overfit runs')

if overfit_losses:
    drop = overfit_losses[0] - overfit_losses[-1]
    pct = drop / overfit_losses[0] * 100
    print(f'\n  Loss: {overfit_losses[0]:.3f} → {overfit_losses[-1]:.3f} ({pct:.0f}% drop)')
    check('Model can overfit (loss drops >40%)', pct > 40,
          f'{pct:.0f}% reduction — learning confirmed')

# Reset model weights for clean training (rebuild)
print('\n  🔄 Rebuilding model with fresh weights for real training...')
torch.manual_seed(42)
model = Lasmoid(args).to(DEVICE)

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 8a: Load TinyStories
# ════════════════════════════════════════════════════════════════
section('PHASE 8: FULL TRAINING — Data Prep')
from datasets import load_dataset

print('  📥 Downloading TinyStories...')
ds = load_dataset('roneneldan/TinyStories', split='train')
ds = ds.select(range(min(30000, len(ds))))

all_tokens = []
for ex in tqdm(ds, desc='Tokenize'):
    t = ex.get('text', '')
    if t and len(t) > 20:
        all_tokens.extend(enc.encode(t) + [EOS_ID])

all_tokens = all_tokens[:len(all_tokens) - len(all_tokens) % SEQ_LEN]
data = torch.tensor(all_tokens, dtype=torch.long)
n_train = int(0.95 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
del all_tokens, ds

BATCH = 8
def get_batch(split='train'):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - SEQ_LEN - 1, (BATCH,))
    xb = torch.stack([d[i:i+SEQ_LEN] for i in ix]).to(DEVICE)
    yb = torch.stack([d[i+1:i+SEQ_LEN+1] for i in ix]).to(DEVICE)
    return xb, yb, torch.ones_like(xb, dtype=torch.float32)

check('Training data ready', len(train_data) > 1_000_000,
      f'{len(train_data)//1000}K train tokens')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 8b: Train 5000 steps
# ════════════════════════════════════════════════════════════════
STEPS = 5000
opts = build_optimizers(model, muon_lr=3e-3, adamw_lr=5e-4, weight_decay=0.1)
scheduler = WSDScheduler(
    opts, warmup_steps=int(0.03*STEPS), stable_steps=int(0.87*STEPS),
    decay_steps=int(0.10*STEPS),
    base_lrs=[[g['lr'] for g in opt.param_groups] for opt in opts],
    min_lr_ratio=0.1,
)

print(f'\n  🚀 Training {STEPS} steps | batch={BATCH} | ~{STEPS*BATCH*SEQ_LEN//1_000_000}M tokens\n')

model.train()
losses, val_losses, val_steps = [], [], []
t0 = time.time()

use_amp = (DEVICE == 'cuda')

for step in range(STEPS):
    scheduler.step(step)
    for opt in opts:
        opt.zero_grad(set_to_none=True)
    xb, yb, mask = get_batch()

    ctx = torch.cuda.amp.autocast(dtype=torch.bfloat16) if use_amp else torch.enable_grad()
    with ctx:
        logits, mtp_logits, _, _, rmaps, _, adjs, eprobs = model(xb, xb)
        loss = compute_loss(logits, yb, rmaps, [model.last_vq_loss], adjs, eprobs,
                            loss_mask=mask, moe_aux_loss=model.last_moe_loss, ignore_index=-100)
        if mtp_logits is not None:
            loss = loss + 0.3 * F.cross_entropy(
                mtp_logits.view(-1, args.vocab_size),
                yb[:, 1:].contiguous().view(-1), ignore_index=-100)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    for opt in opts:
        opt.step()
    losses.append(loss.item())

    # Periodic validation
    if step % 500 == 0:
        model.eval()
        with torch.no_grad():
            vx, vy, vm = get_batch('val')
            vlogits, *_ , vrm, _, vadj, vep = model(vx, vx)
            vloss = compute_loss(vlogits, vy, vrm, [model.last_vq_loss], vadj, vep,
                                 moe_aux_loss=model.last_moe_loss, ignore_index=-100)
        val_losses.append(vloss.item())
        val_steps.append(step)
        model.train()
        if step > 0:
            avg = sum(losses[-500:]) / 500
            elapsed = time.time() - t0
            eta = (STEPS - step) / (step / elapsed) / 60
            print(f'  Step {step:4d}/{STEPS} | train {avg:.3f} | val {vloss.item():.3f} | ETA {eta:.0f}min')

elapsed = (time.time() - t0) / 60
final_train = sum(losses[-100:]) / 100
print(f'\n  ✅ Training done in {elapsed:.0f}min')
print(f'     Train loss: {losses[0]:.2f} → {final_train:.2f}')
print(f'     Val loss:   {val_losses[0]:.2f} → {val_losses[-1]:.2f}')

check('Training loss decreased >30%', (losses[0]-final_train)/losses[0] > 0.3,
      f'{(losses[0]-final_train)/losses[0]*100:.0f}% drop')
check('Val loss decreased', val_losses[-1] < val_losses[0],
      f'{val_losses[0]:.2f} → {val_losses[-1]:.2f}')
check('No overfitting (val close to train)', abs(val_losses[-1] - final_train) < 1.5,
      f'gap={abs(val_losses[-1]-final_train):.2f}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 8c: Plot loss curves
# ════════════════════════════════════════════════════════════════
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    w = 50
    sm = [sum(losses[max(0,i-w):i+1])/len(losses[max(0,i-w):i+1]) for i in range(len(losses))]
    ax.plot(sm, label='train (smoothed)', linewidth=1)
    ax.plot(val_steps, val_losses, 'o-', label='val', color='red', markersize=4)
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title('Lasmoid-10M Training'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'(plot skipped: {e})')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 9: Generation Quality
# ════════════════════════════════════════════════════════════════
section('PHASE 9: GENERATION QUALITY')
from sampler import full_sample

model.eval()

@torch.no_grad()
def generate(prompt, max_tokens=60, temperature=0.7, min_p=0.05):
    tokens = enc.encode(prompt)
    generated = list(tokens)
    pad = SEQ_LEN - len(tokens)
    idx = torch.tensor([([EOS_ID]*pad) + tokens if pad > 0 else tokens[-SEQ_LEN:]],
                       dtype=torch.long, device=DEVICE)
    logits, *_ = model(idx, idx, start_pos=0)
    gen = torch.Generator(device='cpu').manual_seed(42)
    for i in range(max_tokens):
        nid = full_sample(logits[:, -1, :].cpu().float(), generated,
                          temperature=temperature, min_p=min_p, generator=gen)
        tid = nid.item()
        if tid == EOS_ID:
            break
        generated.append(tid)
        inp = torch.tensor([[tid]], dtype=torch.long, device=DEVICE)
        logits, *_ = model(x_enc=None, x_dec=inp, start_pos=SEQ_LEN + i)
    return enc.decode(generated[len(tokens):])

prompts = [
    'Once upon a time, there was a little girl named',
    'The dog ran to the park and',
    'One day, a boy found a magic',
]

outputs = []
for p in prompts:
    o = generate(p)
    outputs.append(o)
    print(f'\n  Prompt: "{p}"')
    print(f'  Output: {o[:200]}')

# Heuristic coherence checks
all_text = ' '.join(outputs).lower()
words = all_text.split()
unique_ratio = len(set(words)) / max(len(words), 1)
avg_word_len = sum(len(w) for w in words) / max(len(words), 1)
has_real_words = sum(1 for w in words if w.isalpha() and 2 <= len(w) <= 12) / max(len(words), 1)

print('\n  Coherence metrics:')
check('Output is non-empty', len(words) > 10, f'{len(words)} words')
check('Vocabulary diversity OK (not repetitive)', 0.15 < unique_ratio < 0.95,
      f'unique ratio={unique_ratio:.2f}')
check('Word lengths look like English (3-7 avg)', 2.5 < avg_word_len < 8,
      f'avg={avg_word_len:.1f} chars')
check('>70% are real-looking words', has_real_words > 0.7,
      f'{has_real_words*100:.0f}%')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 10: FINAL VERDICT
# ════════════════════════════════════════════════════════════════
section('PHASE 10: FINAL VERDICT — GO / NO-GO')

# Categorize checks
CRITICAL = [
    'Forward pass runs', 'Logits shape correct', 'Logits are finite',
    'Backward pass runs', 'Loss is finite', 'No non-finite gradients',
    'Model can overfit (loss drops >40%)', 'Training loss decreased >30%',
    'MoE all experts reachable', 'mHC Sinkhorn doubly-stochastic',
]
QUALITY = [
    'Val loss decreased', 'No overfitting (val close to train)',
    'Vocabulary diversity OK (not repetitive)', '>70% are real-looking words',
    'Word lengths look like English (3-7 avg)',
]

n_total = len(RESULTS)
n_pass = sum(1 for v in RESULTS.values() if v)
crit_pass = all(RESULTS.get(c, False) for c in CRITICAL)
qual_pass = sum(1 for q in QUALITY if RESULTS.get(q, False))

print(f'\n  Overall: {n_pass}/{n_total} checks passed\n')

# List any failures
failures = [k for k, v in RESULTS.items() if not v]
if failures:
    print('  ⚠️  Failed checks:')
    for f in failures:
        tag = '🔴 CRITICAL' if f in CRITICAL else '🟡 quality'
        print(f'      {tag}: {f}')
    print()

print('  ' + '─' * 50)
if crit_pass and qual_pass >= 3:
    print('  🟢🟢🟢  VERDICT: GO  🟢🟢🟢')
    print('  All critical systems work. The architecture is sound.')
    print('  ✅ SAFE TO SCALE to 100M / 300M models.')
    print('\n  Next steps:')
    print('   1. Use notebooks/train_lasmoid_kaggle.ipynb (100M config)')
    print('   2. Train on FineWeb-Edu for 12K+ steps')
    print('   3. Expect coherent output at 100M with real web data')
elif crit_pass:
    print('  🟡🟡🟡  VERDICT: GO WITH CAUTION  🟡🟡🟡')
    print('  Critical systems work but generation quality is weak.')
    print('  This is EXPECTED at 10M — bigger models generate better.')
    print('  ✅ Architecture is validated. Safe to scale.')
    print('  💡 Tip: train longer (10K steps) if you want better 10M output.')
else:
    print('  🔴🔴🔴  VERDICT: NO-GO  🔴🔴🔴')
    print('  Critical systems failed. Do NOT scale yet.')
    print('  Fix the 🔴 CRITICAL failures above before training bigger models.')
print('  ' + '─' * 50)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL: Save checkpoint (optional)
# ════════════════════════════════════════════════════════════════
from safetensors.torch import save_file
save_dir = '/kaggle/working/lasmoid_10m'
os.makedirs(save_dir, exist_ok=True)
save_file(model.state_dict(), os.path.join(save_dir, 'model.safetensors'))
with open(os.path.join(save_dir, 'config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'💾 Saved to {save_dir}')

# Optional HF upload
# from huggingface_hub import HfApi, login, create_repo
# login()  # or use kaggle secret
# api = HfApi(); create_repo('Theory903/lasmoid-10m-test', exist_ok=True)
# api.upload_folder(folder_path=save_dir, repo_id='Theory903/lasmoid-10m-test')